In [1]:
# Identifiers (RQx, Hx, Px, Gx, Fx, Tx) are defined in vpir_bench_spec.md: section 2 for the research questions,
# section 11 for the tests behind each gate, section 12 for phases, gates and figures. Rx reproduction targets are
# in docs/reproduction_targets.md, and numbered decisions in plan.md.
# Planning bound for the Tier 1 grid (make bench or make all), from measurements already taken. Not a result.
# 1 MiB cells: the smoke run's (notebook 14) wall time per (cell, phase), scaled to the spec's repetitions.
# 16 MiB to 1 GiB: notebook 07's per-cell estimates, which measured 32 B and 8 KiB records only, so 256 B and
# 1 KiB take the slower of the two. Cells over the 14 GiB cap (notebook 08) end when the cap kills them.
# S4 from notebook 06: offline cell = 6 digests, online cell = 1 digest + 31 queries.
import json, sys
from collections import defaultdict
from pathlib import Path

repo = Path.cwd().parent
sys.path.insert(0, str(repo / "src"))
from cs4406_sem_project.harness import grid

raw = repo / "results/raw"
MiB = 1 << 20
cells = grid.expand(grid.load(), "tier1")
smoke = {e["cell_id"]: e["wall_s"] for e in map(json.loads, (raw / "20261112T084427Z_smoke_645753/ledger.jsonl").read_text().splitlines())}
est = {(r["arm"], r["db_bytes"], r["record_bytes"]): r
       for r in map(json.loads, (raw / "grid-feasibility-20261111T064956Z/estimates.jsonl").read_text().splitlines())}
ceil = {(r["arm"], r["db_bytes"], r["record_bytes"]): r
        for r in map(json.loads, (raw / "memory-ceiling-20261111T083621Z/ceiling.jsonl").read_text().splitlines())}
s4 = {r["db_bytes"]: r for r in map(json.loads, (raw / "s4-cost-20261110T164450Z/s4_costs.jsonl").read_text().splitlines())}

def minutes(c):
    rec = c.record_bytes or 0
    if c.arm == "S4":
        r = s4[c.db_bytes]
        return ((c.reps + c.warmup) * r["digest_s"] if c.phase == "offline"
                else r["digest_s"] + (c.reps + c.warmup) * r["query_s"]) / 60, "notebook 06"
    if (c.arm, c.db_bytes, rec) in ceil and ceil[(c.arm, c.db_bytes, rec)]["over_cap"]:
        return ceil[(c.arm, c.db_bytes, rec)]["wall_s"] / 60, "oom at 14 GiB (notebook 08)"
    if c.db_bytes == MiB:
        smoke_id = f"{c.arm}_1MiB_{'1bit' if c.arm == 'S3' else '1KiB'}_{c.phase}"
        return smoke[smoke_id] * (c.reps + c.warmup) / 4 / 60, "smoke x reps"
    key = "est_offline_cell_min" if c.phase == "offline" else "est_online_cell_min"
    got = [est[k][key] for k in ((c.arm, c.db_bytes, rec), (c.arm, c.db_bytes, 32), (c.arm, c.db_bytes, 8192)) if k in est and key in est[k]]
    if got:
        return max(got), "notebook 07"
    same = [r for k, r in ceil.items() if k[:2] == (c.arm, c.db_bytes)]
    if all(r["over_cap"] for r in same):  # every measured record size of this arm and size ran out of memory
        return max(r["wall_s"] for r in same) / 60, "oom at 14 GiB (notebook 08, other record sizes)"
    r = ceil[(c.arm, c.db_bytes, rec if (c.arm, c.db_bytes, rec) in ceil else 32)]
    return (c.reps + c.warmup) * r["wall_s"] / 60, "notebook 08, one repetition x reps"

by_source, by_arm, total = defaultdict(float), defaultdict(float), 0.0
rows = []
for c in cells:
    m, src = minutes(c)
    total += m; by_source[src] += m; by_arm[c.arm] += m
    rows.append((m, c.cell_id, src))
print(f"{len(cells)} (cell, phase) runs, planned {total / 60:.1f} h of adapter time")
print("by source:", {k: round(v / 60, 1) for k, v in by_source.items()}, "h")
print("by arm:", {k: round(v / 60, 1) for k, v in sorted(by_arm.items(), key=lambda kv: -kv[1])}, "h")
print("slowest runs:")
for m, cid, src in sorted(rows, reverse=True)[:8]:
    print(f"  {cid:30s} {m:6.1f} min  ({src})")

206 (cell, phase) runs, planned 8.9 h of adapter time
by source: {'smoke x reps': 0.1, 'notebook 07': 6.3, 'oom at 14 GiB (notebook 08)': 1.0, 'oom at 14 GiB (notebook 08, other record sizes)': 0.4, 'notebook 06': 1.1} h
by arm: {'S1': 2.4, 'S2': 2.0, 'S0-cpp': 1.9, 'S4': 1.1, 'S3': 0.8, 'S0-go': 0.6, 'M1': 0.1, 'M0': 0.0} h
slowest runs:
  S4_1MiB_1bit_offline             34.2 min  (notebook 06)
  S2_128MiB_8KiB_offline           19.3 min  (notebook 07)
  S2_128MiB_32B_offline            19.3 min  (notebook 07)
  S2_128MiB_256B_offline           19.3 min  (notebook 07)
  S2_128MiB_1KiB_offline           19.3 min  (notebook 07)
  S1_128MiB_8KiB_offline           19.1 min  (notebook 07)
  S1_128MiB_32B_offline            19.1 min  (notebook 07)
  S1_128MiB_256B_offline           19.1 min  (notebook 07)


In [2]:
# Planning bound for RQ6 and RQ7 (make concurrency): warm-up plus window of every load-generator run at every
# focus cell, before server start-up, drain and the closed sweep's stop rule. The focus cells notebook 08 puts
# over the 14 GiB cap in bench mode are counted apart, since their servers will not fit either. RQ7 is priced
# two ways: configs/load.yaml's 30 runs per W, and 3 runs per W as RQ6 uses.
import copy, yaml
from cs4406_sem_project.harness import loadgen

load = yaml.safe_load((repo / "configs/load.yaml").read_text())
ts = loadgen.targets(grid.load(), "focus")
oom = {(a, d, r) for (a, d, r), v in ceil.items() if v["over_cap"]}
fits_cells = [t for t in ts if (t.arm, t.db_bytes, t.record_bytes or 0) not in oom
              and not (t.arm in ("S1", "S2") and t.db_bytes == 1 << 30)]
print(f"{len(ts)} focus cells, {len(ts) - len(fits_cells)} expected over the cap:",
      sorted(t.cell_id for t in ts if t not in fits_cells))
for runs in (load["multicore"]["reps"], 3):
    l = copy.deepcopy(load); l["multicore"]["reps"] = runs
    rq6 = loadgen.bound_s(l, len(fits_cells), ("closed", "open")) / 3600
    rq7 = loadgen.bound_s(l, len(fits_cells), ("multicore",)) / 3600
    print(f"RQ7 at {runs:2d} runs per W: RQ6 {rq6:.1f} h + RQ7 {rq7:.1f} h = {rq6 + rq7:.1f} h")

26 focus cells, 6 expected over the cap: ['M1_1GiB_32B', 'S1_1GiB_1KiB', 'S1_1GiB_32B', 'S2_1GiB_1KiB', 'S2_1GiB_32B', 'S3_1GiB_1bit']
RQ7 at 30 runs per W: RQ6 16.3 h + RQ7 46.7 h = 63.0 h
RQ7 at  3 runs per W: RQ6 16.3 h + RQ7 4.7 h = 21.0 h
